In [1]:
from dsi.dsi import DSI
from pathlib import Path
import pandas as pd
from pandasql import sqldf

In [8]:
help(DSI)

Help on class DSI in module dsi.dsi:

class DSI(builtins.object)
 |  DSI(filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |
 |  A user-facing interface for DSI's Core middleware.
 |
 |  The DSI Class abstracts Core.Terminal for managing metadata and Core.Sync for data management and movement.
 |
 |  Methods defined here:
 |
 |  __init__(self, filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |      Initializes DSI by activating a backend for data operations; default is a Sqlite backend for temporary data analysis.
 |      If users specify `filename`, data is saved to a permanent backend file.
 |      Can now call read(), find(), update(), query(), write() or any backend printing operations
 |
 |      `filename` : str, optional
 |          If not specified, a temporary, hidden backend file is created for users to analyze their data.
 |          If specified and backend file already exists, it is activated for a user to explore its data.
 |          If specified and ba

In [84]:
class f_dsi:
    def __init__(self, folder_name:str):
        self.folder_name = folder_name
        folder = Path(folder_name)
        db_path_list = []
        for p in folder.iterdir():
            if p.is_file():
                db_path_list.append(p)
        
        databases = []
        for d_id, db_path in enumerate(db_path_list):
            db_info = {}
            try:
                _temp = DSI(str(db_path), silence_messages="True")
                db_info['id'] = d_id
                db_info['path'] = str(db_path)
                db_info['name'] = (str(db_path)).split('/')[-1]
                _tbls = _temp.list(True)
                db_info['num_tables'] = len(_tbls)
                db_info['tables'] = _tbls
                
                _temp.close()
                
                databases.append(db_info)
            except Exception as e:
                print(f"Error {e} opening database at {db_path}!")
    
        self.df = f_collect_databases(folder_name)
        self.df_exp = df.explode("tables").rename(columns={"tables": "table"})

        
    def f_get_databases(self):
        return self.df
    
    def f_summarize(self, table, db):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
        print(path_str)
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.summary(collection=True)
        return l


    def f_query(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.query(query, collection=True)
        return l

    def f_search(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.search(query, collection=True)
        return l

    def f_find(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.find(query, collection=True)
        return l

In [93]:
folder_name = "/home/pascalgrosset/projects/dsi/tools/federated/_workspace_9f7b4fd4/"

In [94]:
x = f_dsi(folder_name)

In [95]:
df = x.f_get_databases()
df

,id,path,name,num_tables,tables
0,0,/home/pascalgrosset/projects/dsi/tools/federat...,modelcard.db,1,[models]
1,1,/home/pascalgrosset/projects/dsi/tools/federat...,ocean_11_datasets.db,1,[genesis_datacard]
2,2,/home/pascalgrosset/projects/dsi/tools/federat...,nif.db,2,"[array_and_types, nif_metadata]"


In [87]:
 x.f_summarize(table="nif_metadata", db="nif.db")

SELECT path FROM df_exp WHERE "table" = 'nif_metadata' AND name = 'nif.db'
/home/pascalgrosset/projects/dsi/tools/federated/_workspace_9f7b4fd4/nif.db


[       column     type unique   min   max   avg std_dev
 0  array_name  VARCHAR      9  None  None  None    None
 1  array_type  VARCHAR      1  None  None  None    None,
         column     type unique            min                max  \
 0     sim_name  VARCHAR     20           None               None   
 1     timestep      INT     20           None               None   
 2   num_arrays      INT      1           None               None   
 3        shape  VARCHAR     19           None               None   
 4     ablt_min    FLOAT      1            0.0                0.0   
 5     ablt_max    FLOAT      1            1.0                1.0   
 6     cham_min    FLOAT      1            0.0                0.0   
 7     cham_max    FLOAT      1            1.0                1.0   
 8     dens_min    FLOAT     17       0.000007           0.000017   
 9     dens_max    FLOAT     20        2.63112           7.954777   
 10    depo_min    FLOAT      1            0.0                0.0   


In [89]:
x.f_find(table="nif_metadata", db="nif.db", query="dens_max > 5")

SELECT path FROM df_exp WHERE "table" = 'nif_metadata' AND name = 'nif.db'
Finding all rows where 'dens_max > 5' in the active backend


,sim_name,timestep,num_arrays,shape,ablt_min,ablt_max,cham_min,cham_max,dens_min,dens_max,...,foam_max,mark_min,mark_max,pres_min,pres_max,tele_min,tele_max,tion_min,tion_max,link
0,nifcylxyz_hdf5_plt_cnt_0005,5,9,"(8664, 16, 16, 16)",0.0,1.0,0.0,1.0,0.000009,5.666894,...,1.0,0.0,1.000000,65130.90625,3.425035e+13,100.00000,23783628.0,148.416901,81353384.0,https://oceans11.lanl.gov/nif/N210201-001_3D_r...
1,nifcylxyz_hdf5_plt_cnt_0007,7,9,"(8304, 16, 16, 16)",0.0,1.0,0.0,1.0,0.000010,7.381287,...,1.0,0.0,0.910547,65130.90625,3.571370e+13,100.00000,15957877.0,202.622498,46893368.0,https://oceans11.lanl.gov/nif/N210201-001_3D_r...
2,nifcylxyz_hdf5_plt_cnt_0006,6,9,"(8408, 16, 16, 16)",0.0,1.0,0.0,1.0,0.000010,7.954777,...,1.0,0.0,0.999962,65130.90625,4.037818e+13,156.77063,23426554.0,156.770630,50003680.0,https://oceans11.lanl.gov/nif/N210201-001_3D_r...
3,nifcylxyz_hdf5_plt_cnt_0008,8,9,"(8496, 16, 16, 16)",0.0,1.0,0.0,1.0,0.000010,5.255617,...,1.0,0.0,0.874049,65130.90625,2.591218e+13,100.00000,11094333.0,204.019913,11615710.0,https://oceans11.lanl.gov/nif/N210201-001_3D_r...
